# An Illustration of MaxSim Latency

In [2]:
import torch
import time

# Ablate these hyperparameters to simulate different query scenarios

NUM_QUERIES = 1
QUERY_VECTORS = 100 # Simulating a longer query
DOC_VECTORS = 1000 # The # of vectors used to represent each document in the database.
DIM = 128 # Dimension per query and document vector
NUM_DOCS = 1000 # Reranking 1,000 candidate documents

In [10]:
# This is an illustration with 1 query.
# However, you could also batch queries in a single GPU inference to further make the case for GPU inference...

def maxsim_batch(queries, docs):
    """
    queries: (Q, Qt, D)
    docs:    (N, Dt, D)
    returns: (Q, N) score matrix
    """
    # Dot Products
    # (Q, 1, Qt, D) @ (1, N, D, Dt) -> (Q, N, Qt, Dt)
    sim = queries[:, None, :, :] @ docs[None, :, :, :].transpose(-1, -2) # `@` is an overloaded matrix multiplication operator in PyTorch...

    # MaxSim
    max_sim = sim.max(dim=-1).values  # (Q, N, Qt) --> The maximum similarity across all documnet tokens

    # Aggregate MaxSim scores
    scores = max_sim.sum(dim=-1)      # (Q, N)

    return scores

In [11]:
# Simluate embeddings with random matrices

def build_data(device, dtype=torch.float32):
    queries = torch.randn(NUM_QUERIES, QUERY_VECTORS, DIM, device=device, dtype=dtype)
    docs = torch.randn(NUM_DOCS, DOC_VECTORS, DIM, device=device, dtype=dtype)
    queries = torch.nn.functional.normalize(queries, dim=-1)
    docs = torch.nn.functional.normalize(docs, dim=-1)
    return queries, docs

In [12]:
def benchmark(device_name, num_warmup=3, num_runs=10):
    device = torch.device(device_name)
    queries, docs = build_data(device)

    for _ in range(num_warmup):
        scores = maxsim_batch(queries, docs)
        if device.type == "cuda":
            torch.cuda.synchronize()

    times = []
    for _ in range(num_runs):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        scores = maxsim_batch(queries, docs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    avg_ms = sum(times) / len(times) * 1000
    min_ms = min(times) * 1000
    return scores, avg_ms, min_ms

In [13]:
print("Running MaxSim on CPU...")
cpu_scores, cpu_avg, cpu_min = benchmark("cpu")
print(f"Score matrix: {cpu_scores.shape}")
print(f"Avg: {cpu_avg:.2f} ms | Best: {cpu_min:.2f} ms")

Running MaxSim on CPU...
Score matrix: torch.Size([1, 1000])
Avg: 162.91 ms | Best: 129.70 ms


In [5]:
!nvidia-smi

Tue Mar 17 23:20:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P0             51W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [14]:
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Running MaxSim on GPU ({gpu_name})...")
    gpu_scores, gpu_avg, gpu_min = benchmark("cuda")
    print(f"Avg: {gpu_avg:.2f} ms | Best: {gpu_min:.2f} ms")
    print(f"Speedup: {cpu_avg / gpu_avg:.1f}x (avg) | {cpu_min / gpu_min:.1f}x (best)")
else:
    print("CUDA not available — try Colab with a GPU runtime.")

Running MaxSim on GPU (NVIDIA A100-SXM4-40GB)...
Avg: 2.82 ms | Best: 2.81 ms
Speedup: 57.8x (avg) | 46.2x (best)


# An Illustration of MUVERA Hashing

In [20]:
import torch

random_doc_embeddings = []
for _ in range(20):
    # You can adjust the dimensions (e.g., 3, 3) as needed
    random_doc_embeddings.append(torch.randn(DOC_VECTORS, DIM))


import torch.nn.functional as F

random_query_embeddings = torch.randn(QUERY_VECTORS, DIM).unsqueeze(0)
random_query_embeddings = F.normalize(random_query_embeddings, dim=-1)

# Make docs 0 and 5 genuinely similar to the query
random_doc_embeddings = [F.normalize(doc, dim=-1) for doc in random_doc_embeddings]

for i in [0, 5]:
    noise = torch.randn(DOC_VECTORS, DIM) * 0.3
    query_expanded = random_query_embeddings.squeeze(0).repeat(
        (DOC_VECTORS // QUERY_VECTORS) + 1, 1
    )[:DOC_VECTORS]
    random_doc_embeddings[i] = F.normalize(query_expanded + noise, dim=-1)

random_doc_embeddings = torch.stack(random_doc_embeddings)


In [21]:
def maxsim_batch(queries, docs):
    """
    queries: (Q, Qt, D)
    docs:    (N, Dt, D)
    returns: (Q, N) score matrix
    """
    # Dot Products
    # (Q, 1, Qt, D) @ (1, N, D, Dt) -> (Q, N, Qt, Dt)
    sim = queries[:, None, :, :] @ docs[None, :, :, :].transpose(-1, -2) # `@` is an overloaded matrix multiplication operator in PyTorch...

    # MaxSim
    max_sim = sim.max(dim=-1).values  # (Q, N, Qt) --> The maximum similarity across all documnet tokens

    # Aggregate MaxSim scores
    scores = max_sim.sum(dim=-1)      # (Q, N)

    return scores

maxsim_batch(random_query_embeddings, random_doc_embeddings)

tensor([[40.8319, 28.5143, 27.7361, 28.1053, 28.1665, 40.2286, 28.5197, 27.8849,
         28.1714, 28.0757, 28.7640, 28.0604, 28.1396, 27.9640, 28.0492, 28.1616,
         28.0754, 28.3561, 27.7485, 28.6792]])

In [23]:
def compute_fde(embeddings, num_buckets=32, num_repetitions=4, hash_projections=None):
    """
    Compute a MUVERA Fixed Dimensional Encoding for a set of token embeddings.

    embeddings:       (T, D) - token embeddings for one document
    num_buckets:      B - number of buckets per repetition
    num_repetitions:  R - number of independent hash repetitions
    hash_projections: list of R random matrices (D, B) for hashing

    returns: (R * B * D,) - fixed-length FDE vector
    """
    D = embeddings.shape[1]

    # Create hash projections if not provided
    if hash_projections is None:
        hash_projections = [torch.randn(D, num_buckets) for _ in range(num_repetitions)]

    parts = []
    for r in range(num_repetitions):
        # Hash each token into a bucket via random projection
        bucket_assignments = (embeddings @ hash_projections[r]).argmax(dim=-1)  # (T,)

        # Average embeddings within each bucket
        bucket_vectors = torch.zeros(num_buckets, D)
        for b in range(num_buckets):
            mask = bucket_assignments == b
            if mask.any():
                bucket_vectors[b] = embeddings[mask].mean(dim=0)

        parts.append(bucket_vectors.reshape(-1))  # (B * D,)

    return torch.cat(parts)  # (R * B * D,)


# --- Build FDEs for all docs ---
NUM_BUCKETS = 32
NUM_REPETITIONS = 4

# Important: share the same hash projections across all docs and queries
hash_projections = [torch.randn(DIM, NUM_BUCKETS) for _ in range(NUM_REPETITIONS)]

doc_fdes = torch.stack([
    compute_fde(random_doc_embeddings[i], NUM_BUCKETS, NUM_REPETITIONS, hash_projections)
    for i in range(random_doc_embeddings.shape[0])
])

# Normalize FDEs for cosine similarity
doc_fdes = F.normalize(doc_fdes, dim=-1)

print(f"Doc embeddings shape:  {random_doc_embeddings.shape}")
print(f"FDE shape per doc:     ({NUM_REPETITIONS} * {NUM_BUCKETS} * {DIM},) = ({doc_fdes.shape[1]},)")
print(f"All doc FDEs shape:    {doc_fdes.shape}")

Doc embeddings shape:  torch.Size([20, 1000, 128])
FDE shape per doc:     (4 * 32 * 128,) = (16384,)
All doc FDEs shape:    torch.Size([20, 16384])


In [32]:
# --- Compute query FDE (same hash projections!) ---
query_fde = compute_fde(random_query_embeddings.squeeze(0), NUM_BUCKETS, NUM_REPETITIONS, hash_projections)
query_fde = F.normalize(query_fde.unsqueeze(0), dim=-1)  # (1, R*B*D)

# --- Cosine similarities between query FDE and all doc FDEs ---
fde_cosine_scores = (query_fde @ doc_fdes.T).squeeze(0)  # (N,)

# --- Compare rankings ---
maxsim_scores = maxsim_batch(random_query_embeddings, random_doc_embeddings).squeeze(0)

maxsim_ranking = maxsim_scores.argsort(descending=True)
fde_ranking = fde_cosine_scores.argsort(descending=True)

print("=" * 60)
print(f"  {'Rank':<6} {'MaxSim':>10} {'idx':>5}   {'FDE cos':>10} {'idx':>5}")
print("=" * 60)
for rank in range(len(maxsim_ranking)):
    ms_idx = maxsim_ranking[rank].item()
    fde_idx = fde_ranking[rank].item()
    ms_score = maxsim_scores[ms_idx].item()
    fde_score = fde_cosine_scores[fde_idx].item()

    print(f"  {rank+1:<6} {ms_score:>10.4f} [{ms_idx:>3}]{ms_marker:<1} {fde_score:>10.4f} [{fde_idx:>5}]{fde_marker}")

  Rank       MaxSim   idx      FDE cos   idx
  1         40.8319 [  0]      0.2474 [    0]
  2         40.2286 [  5]      0.2411 [    5]
  3         28.7640 [ 10]      0.2044 [   13]
  4         28.6792 [ 19]      0.2014 [   10]
  5         28.5197 [  6]      0.2003 [    8]
  6         28.5143 [  1]      0.2000 [    3]
  7         28.3561 [ 17]      0.1998 [   19]
  8         28.1714 [  8]      0.1997 [   11]
  9         28.1665 [  4]      0.1984 [   12]
  10        28.1616 [ 15]      0.1983 [   15]
  11        28.1396 [ 12]      0.1981 [   16]
  12        28.1053 [  3]      0.1978 [   17]
  13        28.0757 [  9]      0.1974 [    4]
  14        28.0754 [ 16]      0.1968 [    9]
  15        28.0604 [ 11]      0.1963 [    2]
  16        28.0492 [ 14]      0.1955 [    6]
  17        27.9640 [ 13]      0.1943 [    1]
  18        27.8849 [  7]      0.1913 [   14]
  19        27.7485 [ 18]      0.1903 [    7]
  20        27.7361 [  2]      0.1877 [   18]
